# Experiment 1 — the same query as SQL and as DataFrame API

Spark UI: http://localhost:4040. Each **PREDICT** prompt is answered in chat before running the cell.

In [11]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName("m2-exp1-sql-vs-df").getOrCreate()

orders = spark.read.parquet("/opt/data/m1/orders")
customers = spark.read.parquet("/opt/data/m1/customers")
orders.createOrReplaceTempView("orders")
customers.createOrReplaceTempView("customers")

## A — DataFrame API (the Module 1 pipeline)

In [12]:
df_a = (
    orders.filter(F.col("order_date") >= "2026-06-01")
    .groupBy("customer_id")
    .agg(F.sum("amount").alias("total"), F.count("*").alias("n"))
    .join(customers, "customer_id")
    .select("customer_id", "country", "segment", "total", "n")
)

## B — the same query as SQL, same shape (aggregate first, then join)

**PREDICT:** will `df_b` have the same physical plan as `df_a`? Same operators, same `Exchange` placement, same join?

In [13]:
df_b = spark.sql("""
    WITH agg AS (
        SELECT customer_id, sum(amount) AS total, count(*) AS n
        FROM orders
        WHERE order_date >= '2026-06-01'
        GROUP BY customer_id
    )
    SELECT a.customer_id, c.country, c.segment, a.total, a.n
    FROM agg a JOIN customers c ON a.customer_id = c.customer_id
""")

In [14]:
df_a.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (12)
+- Project (11)
   +- BroadcastHashJoin Inner BuildRight (10)
      :- HashAggregate (6)
      :  +- Exchange (5)
      :     +- HashAggregate (4)
      :        +- Project (3)
      :           +- Filter (2)
      :              +- Scan parquet  (1)
      +- BroadcastExchange (9)
         +- Filter (8)
            +- Scan parquet  (7)


(1) Scan parquet 
Output [3]: [customer_id#67, amount#69, order_date#70]
Batched: true
Location: InMemoryFileIndex [file:/opt/data/m1/orders]
PushedFilters: [IsNotNull(order_date), GreaterThanOrEqual(order_date,2026-06-01), IsNotNull(customer_id)]
ReadSchema: struct<customer_id:int,amount:double,order_date:date>

(2) Filter
Input [3]: [customer_id#67, amount#69, order_date#70]
Condition : ((isnotnull(order_date#70) AND (order_date#70 >= 2026-06-01)) AND isnotnull(customer_id#67))

(3) Project
Output [2]: [customer_id#67, amount#69]
Input [3]: [customer_id#67, amount#69, order_date#70]

(4) HashAggregate
Input 

In [15]:
df_b.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (12)
+- Project (11)
   +- BroadcastHashJoin Inner BuildRight (10)
      :- HashAggregate (6)
      :  +- Exchange (5)
      :     +- HashAggregate (4)
      :        +- Project (3)
      :           +- Filter (2)
      :              +- Scan parquet  (1)
      +- BroadcastExchange (9)
         +- Filter (8)
            +- Scan parquet  (7)


(1) Scan parquet 
Output [3]: [customer_id#67, amount#69, order_date#70]
Batched: true
Location: InMemoryFileIndex [file:/opt/data/m1/orders]
PushedFilters: [IsNotNull(order_date), GreaterThanOrEqual(order_date,2026-06-01), IsNotNull(customer_id)]
ReadSchema: struct<customer_id:int,amount:double,order_date:date>

(2) Filter
Input [3]: [customer_id#67, amount#69, order_date#70]
Condition : ((isnotnull(order_date#70) AND (order_date#70 >= 2026-06-01)) AND isnotnull(customer_id#67))

(3) Project
Output [2]: [customer_id#67, amount#69]
Input [3]: [customer_id#67, amount#69, order_date#70]

(4) HashAggregate
Input 

Spark can compare two plans itself (after normalising aliases and ids):

In [16]:
df_a.sameSemantics(df_b)

False

## C — the same *result* written the natural SQL way (join first, then group by everything)

**PREDICT:** does Catalyst rewrite this into the aggregate-first plan of A/B? Same physical plan, or a different one — and if different, where?

In [17]:
df_c = spark.sql("""
    SELECT o.customer_id, c.country, c.segment, sum(o.amount) AS total, count(*) AS n
    FROM orders o JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.order_date >= '2026-06-01'
    GROUP BY o.customer_id, c.country, c.segment
""")
df_c.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (12)
+- HashAggregate (11)
   +- Exchange (10)
      +- HashAggregate (9)
         +- Project (8)
            +- BroadcastHashJoin Inner BuildRight (7)
               :- Project (3)
               :  +- Filter (2)
               :     +- Scan parquet  (1)
               +- BroadcastExchange (6)
                  +- Filter (5)
                     +- Scan parquet  (4)


(1) Scan parquet 
Output [3]: [customer_id#67, amount#69, order_date#70]
Batched: true
Location: InMemoryFileIndex [file:/opt/data/m1/orders]
PushedFilters: [IsNotNull(order_date), GreaterThanOrEqual(order_date,2026-06-01), IsNotNull(customer_id)]
ReadSchema: struct<customer_id:int,amount:double,order_date:date>

(2) Filter
Input [3]: [customer_id#67, amount#69, order_date#70]
Condition : ((isnotnull(order_date#70) AND (order_date#70 >= 2026-06-01)) AND isnotnull(customer_id#67))

(3) Project
Output [2]: [customer_id#67, amount#69]
Input [3]: [customer_id#67, amount#69, order_date#70

In [18]:
df_a.sameSemantics(df_c)

False

## Run all three and compare in the SQL / DataFrame tab

Look at the number of rows entering the join node in A/B vs C.

In [19]:
print(df_a.count(), df_b.count(), df_c.count())

10000 10000 10000


26/09/20 13:23:34 WARN JavaUtils: Attempt to delete using native Unix OS command failed for path = /tmp/spark-a0051b64-972e-4b7f-810f-6a30b067e043/pyspark-918def07-be0e-401e-8409-9c701e1f724f. Falling back to Java IO way
java.io.IOException: Failed to delete: /tmp/spark-a0051b64-972e-4b7f-810f-6a30b067e043/pyspark-918def07-be0e-401e-8409-9c701e1f724f
	at org.apache.spark.network.util.JavaUtils.deleteRecursivelyUsingUnixNative(JavaUtils.java:352)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:269)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:248)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively(SparkFileUtils.scala:158)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively$(SparkFileUtils.scala:157)
	at org.apache.spark.util.Utils$.deleteRecursively(Utils.scala:1062)
	at org.apache.spark.util.ShutdownHookManager$.$anonfun$new$4(ShutdownHookManager.scala:70)
	at org.apache.spark.util.ShutdownHookManager$.$anonfu

## Stop before opening the next notebook

In [10]:
spark.stop()